In [0]:
-- ==============================================================================
-- PROJETO 2 - FASE 3: MÉTRICAS FINANCEIRAS (CAMADA GOLD COM SPARK SQL)
-- ==============================================================================

CREATE OR REPLACE TABLE workspace.crypto_analytics.gold_crypto_dashboard AS
WITH cte_ultimas_cotacoes AS (
    SELECT 
        nome,
        simbolo,
        preco_atual_usd,
        valor_de_mercado_usd,
        volume_negociado_24h,
        variacao_percentual_24h,
        data_hora_extracao,
        -- Window Function para identificar a linha mais recente de cada moeda
        ROW_NUMBER() OVER (PARTITION BY simbolo ORDER BY data_hora_extracao DESC) AS linha_recente,
        -- média histórica de todas as capturas desta moeda até agora
        AVG(preco_atual_usd) OVER (PARTITION BY simbolo) AS preco_medio_historico
    FROM workspace.crypto_analytics.silver_crypto
)
SELECT 
    nome,
    simbolo,
    preco_atual_usd AS ultimo_preco_usd,
    ROUND(preco_medio_historico, 2) AS preco_medio_historico_usd,
    
    -- Regra de Negócio: Avaliação de Risco baseada na variação das últimas 24h
    CASE 
        WHEN ABS(variacao_percentual_24h) > 5.0 THEN 'Alta Volatilidade (Risco Alto)'
        WHEN ABS(variacao_percentual_24h) BETWEEN 2.0 AND 5.0 THEN 'Volatilidade Moderada (Risco Médio)'
        ELSE 'Estável (Risco Baixo)'
    END AS classificacao_risco,
    
    ROUND(valor_de_mercado_usd, 2) AS capitalizacao_mercado_usd,
    ROUND(volume_negociado_24h, 2) AS volume_24h_usd,
    data_hora_extracao AS ultima_atualizacao_perfil
FROM cte_ultimas_cotacoes
WHERE linha_recente = 1 -- Filtra apenas a "fotografia" mais recente do mercado
ORDER BY capitalizacao_mercado_usd DESC;